# 长沙二手房数据分析

## 项目背景
基于长沙二手房真实挂牌数据，分析各区域房价水平、房屋品质与价格的关系，辅助购房决策。

## 数据说明
- 数据源：house_data.csv（地区、总价、单价、总面积、建造时间、楼层、梯户比例、装修情况等字段）
- 工具：Python + pandas

## 分析流程
1. 数据读取与质量检测（缺失值、重复值）
2. 区域房价分析（总价/单价中位数排名）
3. 特征筛选（高单价小户型、老破小、刚需房）
4. 品质溢价分析（精装溢价率、次新房溢价）
5. 梯户比分类与密度分析


In [1]:
# ===== 导入库 =====
# pandas：数据分析核心库
# numpy：数值计算库
import pandas as pd
import numpy as np

In [2]:
# ===== 读取数据 =====
# 读取长沙二手房数据 CSV
# head(10)：预览前 10 行
Real = pd.read_csv('D:/浏览器杂项/house_data.csv')
Real.head(10)

,总价(万),单价(元/平米),房型,楼层,朝向,户型结构,装修情况,总面积,建造时间,小区名称,地区,是否有电梯,建筑结构,梯户比例
0,145.0,11241,3室2厅2卫,中楼层,南北,平层,其他,129.00,2013,湘江700,岳麓,有,钢混结构,其他
1,55.0,5842,3室2厅2卫,中楼层,东南,平层,简装,94.15,2013,才子城,望城,有,钢混结构,两梯四户
2,140.0,13483,3室2厅1卫,低楼层,南,平层,精装,103.84,2010,星城荣域,天心,有,钢混结构,两梯四户
3,141.0,10470,3室2厅2卫,低楼层,南北,平层,精装,134.68,2006,富景园,天心,有,钢混结构,一梯两户
4,236.0,18386,4室2厅2卫,高楼层,南,平层,精装,128.36,2016,旭辉国际广场,雨花,有,钢混结构,一梯两户
5,89.8,6380,4室2厅2卫,低楼层,南北,平层,毛坯,140.76,2013,荣盛城,长沙县,有,钢混结构,其他
6,92.8,10533,3室2厅1卫,高楼层,南北,平层,精装,88.11,2014,芒果雅苑,雨花,有,钢混结构,两梯六户
7,162.0,13858,3室2厅2卫,中楼层,东南,平层,毛坯,116.90,2013,中建钰和城,天心,有,钢混结构,其他
8,18.0,3624,2室1厅1卫,低楼层,南,平层,毛坯,49.68,2013,大汉月亮河畔,望城,有,钢混结构,六梯二十四户
9,52.8,5725,2室2厅1卫,低楼层,南,平层,其他,92.24,2013,鑫天山城明珠,天心,有,钢混结构,其他


In [3]:
# ===== 数据探查 =====
# size：总元素个数；columns：列名；dtypes：各列数据类型
print('--------------------该数据表的大致情况--------------------')
print('该数据表的大小:\n',Real.size)
print(' ')
print('该数据表的所有列:\n',Real.columns)
print(' ')
print('该数据表的每个列是什么数据类型?\n',Real.dtypes)

--------------------该数据表的大致情况--------------------
该数据表的大小:
 21000
 
该数据表的所有列:
 Index(['总价(万)', '单价(元/平米)', '房型', '楼层', '朝向', '户型结构', '装修情况', '总面积', '建造时间',
       '小区名称', '地区', '是否有电梯', '建筑结构', '梯户比例'],
      dtype='str')
 
该数据表的每个列是什么数据类型?
 总价(万)       float64
单价(元/平米)      int64
房型              str
楼层              str
朝向              str
户型结构            str
装修情况            str
总面积         float64
建造时间          int64
小区名称            str
地区              str
是否有电梯           str
建筑结构            str
梯户比例            str
dtype: object


In [4]:
# ===== 缺失值与重复值检测 =====
# isnull().sum()：各列缺失数量；duplicated().sum()：重复行数
print('--------------------该数据表的缺失值与重复值检测--------------------')
print('该数据表的缺失值检测结果如下:\n',Real.isnull().sum())
print(' ')
print('该数据表的重复值检测结果如下:\n',Real.duplicated().sum())

--------------------该数据表的缺失值与重复值检测--------------------
该数据表的缺失值检测结果如下:
 总价(万)       0
单价(元/平米)    0
房型          0
楼层          0
朝向          0
户型结构        0
装修情况        0
总面积         0
建造时间        0
小区名称        0
地区          0
是否有电梯       0
建筑结构        0
梯户比例        0
dtype: int64
 
该数据表的重复值检测结果如下:
 5


In [5]:
# ===== 重复值清洗 =====
# drop_duplicates(inplace=True)：删除完全重复的行
# 清洗后再次检测确认
print('--------------------该数据表的重复值清洗--------------------')
Real.drop_duplicates(inplace=True)
print('执行命令中zzzzzzz~')
print('执行完成!')
print(' ')
print('再次检测重复值!')
print('重复值检测结果如下:\n',Real.duplicated().sum())

--------------------该数据表的重复值清洗--------------------
执行命令中zzzzzzz~
执行完成!
 
再次检测重复值!
重复值检测结果如下:
 0


In [6]:
# ===== 区域房价分析 =====
# 筛选长沙主要城区，分别计算总价/单价中位数（中位数抗极端值干扰）
# 单价降序排名 + merge 拼接两个结果
area = ['岳麓','雨花','望城','天心','开福','芙蓉','长沙县']
area_boll = Real[(Real['地区'].isin(area))]
Total_Price_Median = area_boll.groupby('地区',as_index=False)['总价(万)'].median().rename(columns={'总价(万)':'总价的中位数'})
Unit_Price_Median = area_boll.groupby('地区',as_index=False)['单价(元/平米)'].median().rename(columns={'单价(元/平米)':'单价的中位数'})
Unit_Price_Ranking = Unit_Price_Median.sort_values('单价的中位数',ascending=False)
merge_data = pd.merge(Total_Price_Median,Unit_Price_Median,on='地区')
merge_data

,地区,总价的中位数,单价的中位数
0,天心,105.00,10470.0
1,岳麓,130.00,11642.5
2,开福,103.50,9975.5
3,望城,83.68,7631.0
4,芙蓉,98.00,9889.0
5,长沙县,75.00,7419.0
6,雨花,108.00,9967.0


In [7]:
# ===== 高单价小户型筛选 =====
# 条件：单价≥区域中位数1.5倍 且 总价≤区域中位数 且 面积<90㎡
# 找出'单价虚高的小户型'（whz 变量）
Real_2 = pd.merge(Real,merge_data,on='地区',how='left')
whz = Real_2 [(Real_2['单价(元/平米)'] >= Real_2['单价的中位数']*1.5) & 
    (Real_2['总价(万)'] <= Real_2['总价的中位数']) & 
     (Real_2['总面积'] < 90)]
whz

,总价(万),单价(元/平米),房型,楼层,朝向,户型结构,装修情况,总面积,建造时间,小区名称,地区,是否有电梯,建筑结构,梯户比例,总价的中位数,单价的中位数
517,50.0,11735,1室1厅1卫,高楼层,西北,平层,其他,42.61,2018,乾源国际广场,望城,有,钢混结构,两梯五户,83.68,7631.0


In [8]:
# ===== 老破小与电梯统计 =====
# 老破小：建造时间≤2005 且 面积<70㎡，按总价升序
# 统计有电梯房源数量与单价中位数
old_house = Real.query('建造时间 <= 2005 and 总面积 <70')[['小区名称','地区','建造时间','总面积','总价(万)']].sort_values('总价(万)')
Elevator_Statistics_True = Real.query('是否有电梯 == "有"')['是否有电梯'].count()
Elevator_Statistics_True_median = Real.query('是否有电梯 == "有"')['单价(元/平米)'].median()
print(f"有电梯{Elevator_Statistics_True}套,中位数单价{Elevator_Statistics_True_median},无电梯0套，样本数量不足")

有电梯1495套,中位数单价9288.0,无电梯0套，样本数量不足


In [9]:
# 查看老破小明细
old_house

,小区名称,地区,建造时间,总面积,总价(万)
332,朝阳二村,芙蓉,1985,46.48,21.0
337,登隆街社区,芙蓉,1990,42.60,25.0
954,狮子山一片小区,雨花,2004,54.90,32.0
62,省金属回收公司宿舍,芙蓉,1995,68.51,36.8
27,信和苑一区,雨花,2004,64.76,37.0
1267,中国冶金地质总局湖南地质勘察院,芙蓉,1995,61.60,40.0
185,长铁向韶村,芙蓉,1989,69.59,60.0
1062,湖南省人民政府机关梓园住宅区,雨花,1985,61.01,76.0


In [10]:
# 查看无电梯房源（样本量对比）
Elevator_Statistics_True = Real.query('是否有电梯 == "无"')['是否有电梯']
Elevator_Statistics_True

Series([], Name: 是否有电梯, dtype: str)

In [11]:
# ===== 装修溢价率分析 =====
# 分别计算精装/毛坯在各区域的中位数单价
# 溢价率 = (精装-毛坯)/毛坯 × 100%
Deluxe = Real.query('装修情况 == "精装"')
bare_shell = Real.query('装修情况 == "毛坯"')
Deluxe_group = Deluxe.groupby('地区',as_index=False)['单价(元/平米)'].median().rename(columns={'单价(元/平米)':'单价中位数_Deluxe'})
bare_shell_group = bare_shell.groupby('地区',as_index=False)['单价(元/平米)'].median().rename(columns={'单价(元/平米)':'单价中位数_bare_shell'})
Premium_Calculation = pd.merge(Deluxe_group,bare_shell_group,on='地区')
Premium_Calculation['溢价率'] = ((Premium_Calculation['单价中位数_Deluxe'] - Premium_Calculation['单价中位数_bare_shell'])/Premium_Calculation['单价中位数_bare_shell']*100).round(2).astype('str')+'%'
Premium_Calculation

,地区,单价中位数_Deluxe,单价中位数_bare_shell,溢价率
0,天心,10858.0,10004.0,8.54%
1,宁乡,5648.0,4394.0,28.54%
2,岳麓,12445.0,10953.5,13.62%
3,开福,10530.0,9652.0,9.1%
4,望城,8606.0,6843.0,25.76%
5,芙蓉,10079.0,10152.5,-0.72%
6,长沙县,7591.0,7459.0,1.77%
7,雨花,10013.0,10005.5,0.07%


In [12]:
# 查看梯户比例分布（前 60 种）
Real['梯户比例'].value_counts().sort_values(ascending=False).head(60)

梯户比例
两梯四户      557
其他        401
两梯三户      140
两梯两户       62
两梯六户       42
两梯五户       36
一梯两户       34
三梯四户       22
两梯八户       15
三梯六户       11
三梯八户        8
两梯七户        8
六梯二十六户      7
一梯三户        6
四梯四户        5
两梯十六户       5
六梯二十二户      5
三梯三户        4
五梯十七户       4
四梯二十户       4
三梯五户        4
两梯十户        4
一梯四户        4
四梯二十四户      4
四梯十九户       4
三梯十八户       3
两梯九户        3
四梯二十三户      3
四梯十八户       3
三梯两户        3
四梯二十九户      3
三梯十五户       2
四梯五户        2
三梯十户        2
三梯九户        2
两梯二十七户      2
两梯十一户       2
八梯二十七户      2
十三梯七户       2
四梯二十五户      2
五梯二十户       2
四梯八户        2
六梯二十户       2
一梯五户        2
三梯十二户       2
三梯二十三户      2
六梯五十六户      2
八梯二十六户      2
八梯五户        2
六梯二十四户      1
四梯十五户       1
两梯十二户       1
六梯三十户       1
六梯四户        1
四梯十四户       1
四梯三十户       1
四梯十一户       1
五梯四十五户      1
六梯八户        1
五梯十六户       1
Name: count, dtype: int64

In [13]:
# ===== 梯户比分类函数 =====
# 根据'几梯几户'将小区分为低密/中密/高密三类
# 低密：户数少舒适度高；高密：户数多密度高
def classify_ti_hu_ratio(ratio):
    """
    梯户比分类：低密、中密、高密
    """
    # 低密：户数少，舒适度高
    low_density = ['一梯两户', '两梯两户', '两梯三户', '一梯三户', '三梯两户']
    
    # 高密：户数多，密度高
    high_density = [
        '两梯六户', '两梯七户', '两梯八户', '两梯九户', '两梯十户', '两梯十一户',
        '三梯六户', '三梯八户', '三梯九户', '三梯十户', '三梯十五户', '三梯十八户',
        '四梯六户', '四梯十八户', '四梯十九户', '四梯二十户', '四梯二十三户', '四梯二十四户', '四梯二十九户',
        '五梯十七户',
        '六梯二十二户', '六梯二十六户',
        '八梯二十七户',
        '两梯十六户', '两梯二十七户'
    ]
    
    if ratio in low_density:
        return '低密'
    elif ratio in high_density:
        return '高密'
    else:
        return '中密'


In [14]:
# 应用分类函数，新增'梯户划分'列
Real['梯户划分'] = Real['梯户比例'].apply(classify_ti_hu_ratio)

In [15]:
# 不同密度档位的中位数单价对比（低密通常更贵）
apartment_unit = Real.groupby('梯户划分',as_index=False)['单价(元/平米)'].median().sort_values('单价(元/平米)',ascending=False)\
.rename(columns={'单价(元/平米)':'单价中位数'})
apartment_unit     

,梯户划分,单价中位数
1,低密,10530.0
0,中密,9054.5
2,高密,8715.5


In [16]:
# 建造时间分布统计
Real['建造时间'].value_counts()

建造时间
2013    818
2014     72
2016     71
2015     62
2010     61
2012     55
2011     45
2017     45
2018     39
2008     34
2019     31
2009     24
2007     23
2020     21
2021     16
2004     14
2006     13
2005      9
2023      8
2002      7
2022      7
2003      4
1995      4
1990      3
2000      3
1985      2
1989      1
2001      1
1999      1
1996      1
Name: count, dtype: int64

In [17]:
# ===== 房龄划分函数 =====
# ≥2022：全新；≥2016：次新；否则：老旧
def Time_Division(data):
    if data >= 2022:
        return '全新'
    elif data >= 2016:
        return '次新'
    else:
        return '老旧'

In [18]:
# 应用房龄划分，新增'次新房划分'列
Real['次新房划分'] = Real['建造时间'].apply(Time_Division)

In [57]:
# ===== 次新 vs 老旧溢价分析 =====
# 各地区次新房与老旧房的中位数单价差
# ⚠️ 注意：x.loc[3] 用的是行标签定位，若排序后索引不是 0-6 会取错行或报错，建议改用 iloc
old_house = Real.query('次新房划分 == "次新"')
old_house_group = old_house.groupby('地区',as_index=False)['单价(元/平米)'].median()
laojiu = Real.query('次新房划分 == "老旧"')
laojiu_group = laojiu.groupby('地区',as_index=False)['单价(元/平米)'].median()
Discount_rate = pd.merge(old_house_group,laojiu_group,on='地区').rename(columns={'单价(元/平米)_x':'单价(元/平米)_次新','单价(元/平米)_y':'单价(元/平米)_老旧'})
Discount_rate['中位数单价差'] = Discount_rate['单价(元/平米)_次新'] - Discount_rate['单价(元/平米)_老旧']
x = Discount_rate.sort_values('中位数单价差',ascending=False)
max_premium = x.loc[3,'中位数单价差']
min_premium = x.loc[1,'中位数单价差']
print('='*110)
print('次新房溢价分析结果')
print('='*110)
print(f"次新溢价最高{max_premium}")
print(f"次新溢价最低{min_premium }")

次新房溢价分析结果
次新溢价最高3708.0
次新溢价最低-987.0


In [53]:
# 重复计算溢价差（练习性质，逻辑同 cell[18]）
Discount_rate
x = Discount_rate.sort_values('中位数单价差',ascending=False)
max_premium = x.loc[3,'中位数单价差']
x
min_premium = x.loc[0,'中位数单价差']
x

,地区,单价(元/平米)_次新,单价(元/平米)_老旧,中位数单价差
3,开福,13614.0,9906.0,3708.0
5,芙蓉,10960.0,9733.0,1227.0
6,长沙县,7795.0,6792.5,1002.5
7,雨花,10514.0,9657.0,857.0
4,望城,8254.0,7535.0,719.0
0,天心,10694.5,10432.0,262.5
2,岳麓,11020.0,11623.0,-603.0
1,宁乡,4302.0,5289.0,-987.0


In [66]:
# ===== 刚需房筛选 =====
# 条件：总价≤120万、面积70-110㎡、中低楼层、2010年后建造
# 输出前 20 条符合刚需画像的房源
rigid_demand = Real[(Real['总价(万)'] <= 120 ) & 
    (Real['总面积'] >= 70) & (Real['总面积'] <= 110) & 
    (Real['楼层'].isin(['中楼层','低楼层'])) & 
    (Real['建造时间'] >= 2010)]
rigid_demand[['地区','小区名称','总价(万)','单价(元/平米)','总面积','户型结构','楼层']].head(20)

,地区,小区名称,总价(万),单价(元/平米),总面积,户型结构,楼层
1,望城,才子城,55.0,5842,94.15,平层,中楼层
9,天心,鑫天山城明珠,52.8,5725,92.24,平层,低楼层
10,长沙县,保利香槟国际,74.0,8315,89.00,平层,中楼层
16,长沙县,锦璨家园,65.8,6327,104.00,平层,低楼层
19,望城,新华联梦想城,86.0,10125,84.94,平层,中楼层
45,长沙县,国泰九龙湾,117.0,10905,107.30,平层,中楼层
47,天心,湘江锦绣,86.0,8631,99.65,平层,中楼层
50,长沙县,未来康桥长郡,58.0,7113,81.55,平层,低楼层
51,望城,时代倾城,68.0,7786,87.34,平层,中楼层
56,长沙县,长沙雅居乐新地,81.0,8439,95.99,平层,中楼层


In [73]:
# ===== 小区价格波动分析 =====
# std()：计算各小区单价的离散程度（标准差），波动越大越异常
Unit_Price_std = Real.groupby('小区名称',as_index=False)['单价(元/平米)'].std()
Unit_Price_std
Real.query('小区名称 == "龙湖璟宸原著香颂"')

,总价(万),单价(元/平米),房型,楼层,朝向,户型结构,装修情况,总面积,建造时间,小区名称,地区,是否有电梯,建筑结构,梯户比例,梯户划分,次新房划分
445,87.5,8430,3室2厅2卫,高楼层,南,平层,毛坯,103.80,2013,龙湖璟宸原著香颂,望城,有,钢混结构,其他,中密,老旧
904,80.0,7631,3室2厅2卫,中楼层,南,平层,毛坯,104.84,2013,龙湖璟宸原著香颂,望城,有,钢混结构,两梯四户,中密,老旧


In [70]:
# 查看价格波动表
Unit_Price_std

,小区名称,单价(元/平米)
0,CROSS尚公馆,NaN
1,MOMA当代广场,NaN
2,一品东庭,NaN
3,万博汇名邸二期,NaN
4,万国城MOMA一期,NaN
...,...,...
838,龙湖璟宸原著紫宸,1159.750368
839,龙湖璟宸原著香颂,564.978318
840,龙湖碧桂园天宸原著,NaN
841,龙福小区,NaN


In [72]:
# ⚠️ risky_property 变量未定义，运行会 NameError（建议基于 cell[21] 的 std 结果筛选异常小区）
risky_property

,总价(万),单价(元/平米),房型,楼层,朝向,户型结构,装修情况,总面积,建造时间,小区名称,地区,是否有电梯,建筑结构,梯户比例,梯户划分,次新房划分
